In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3


In [ ]:
import hashlib
import time as _clock
NOTEBOOK_START = _clock.time()
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(INPUT.rglob("repo/scripts/omnivoice_adapt.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/omnivoice_adapt.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-omnivoice-train-c-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
CV_DIR, NEY_DIR = Path("/tmp/cv26"), Path("/tmp/neyshekar")
ARTIFACTS = {}
if not GATES.skip("download", ("runner",)):
    start = time.time()
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text("{\"urls\": [{\"name\": \"train_config_finetune_sdpa.json\", \"git_blob_prefix\": \"4b0a543e23be\", \"path\": \"/tmp/adapt/official/train_config_finetune_sdpa.json\", \"url\": \"https://raw.githubusercontent.com/k2-fsa/OmniVoice/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/examples/config/train_config_finetune_sdpa.json\"}], \"snapshots\": [{\"name\": \"base\", \"repo\": \"k2-fsa/OmniVoice\", \"revision\": \"c5fdb5ccb189668d56333f77ba2629f4cd7535f4\", \"local_dir\": \"/tmp/omnivoice-base\", \"sha256\": {\"audio_tokenizer/model.safetensors\": \"fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d\"}}]}")
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *['--no-xet']],
                      timeout=20 * 60, log=LOGS / "fetch.log", monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    if run.ok and report.get("ok") is True:
        ARTIFACTS = {"base": "/tmp/omnivoice-base", "config": "/tmp/adapt/official/train_config_finetune_sdpa.json", "fetched": fetched}
        if Path("/tmp/higgs-audio-v2-tokenizer").is_dir():
            ARTIFACTS["tokenizer"] = "/tmp/higgs-audio-v2-tokenizer"
        GATES.record("download", True, f"{(time.time() - start) / 60:.1f} min; {json.dumps(ARTIFACTS)}")
    else:
        GATES.record("download", False, f"{run.summary()}; fetched so far: {json.dumps(fetched)}")


In [ ]:
# The tokens come from session a's output, the checkpoint from the previous
# session's. data.lst names absolute paths from session a, so it is rewritten
# to where the shards are mounted here - the shards themselves are read as they are.
TOKENS = WORK / "tokens"
RESUME = None
if not GATES.skip("mounted", ("download",)):
    try:
        sources = {}
        for split in ("train", "valid"):
            found = sorted(INPUT.rglob(f"tokens/{split}/data.lst"))
            assert found, f"no tokens/{split}/data.lst under /kaggle/input"
            source = found[0].parent
            lines = []
            for line in found[0].read_text().splitlines():
                tar, jsonl, count, seconds = line.split()
                tar = source / "audios" / Path(tar).name
                jsonl = source / "txts" / Path(jsonl).name
                assert tar.is_file() and jsonl.is_file(), (tar, jsonl)
                lines.append(f"{tar} {jsonl} {count} {seconds}")
            (TOKENS / split).mkdir(parents=True, exist_ok=True)
            (TOKENS / split / "data.lst").write_text("\n".join(lines) + "\n")
            sources[split] = {"shards": len(lines), "from": str(source)}
        checkpoints = sorted(INPUT.rglob("run/checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        checkpoints = [p for p in checkpoints if p.is_dir() and (p / "optimizer.bin").is_file()]
        assert checkpoints, "no checkpoint with optimizer state under /kaggle/input"
        RESUME = checkpoints[-1]
        GATES.record("mounted", True, json.dumps({**sources, "resume_from": str(RESUME)}))
    except Exception as exc:  # noqa: BLE001 - the gate records it
        GATES.record("mounted", False, f"{type(exc).__name__}: {exc}")


In [ ]:
RUN_DIR = Path("/tmp/run")
TRAIN_FACTS = {}
if not GATES.skip("train", ("mounted",)):
    remaining = NOTEBOOK_START + 11.25 * 3600 - time.time()
    if remaining < 2.3 * 3600:
        GATES.record("train", False, f"not started: {remaining / 3600:.2f} h left, less than one checkpoint needs")
    else:
        DATA_CONFIG = WORK / "data_config.json"
        DATA_CONFIG.write_text(json.dumps({"train": [{"manifest_path": [str(TOKENS / "train/data.lst")]}],
                                            "dev": [{"manifest_path": [str(TOKENS / "valid/data.lst")]}]}, indent=2))
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        command = ["accelerate", "launch", "--gpu_ids", "0,1", "--num_processes", "2",
            str(REPO / "scripts/omnivoice_adapt.py"),
            "--train_config", ARTIFACTS["config"], "--data_config", str(DATA_CONFIG), "--output_dir", str(RUN_DIR),
            "--set", f"init_from_checkpoint={ARTIFACTS['base']}", "--set", "keep_last_n_checkpoints=1",
        "--set", f"resume_from_checkpoint={RESUME}",
            "--memory", "checkpointing+bucket-view", "--session-seconds", str(int(remaining - 600)),
            "--export-every", "1000", "--audio-tokenizer", str(Path(ARTIFACTS["base"]) / "audio_tokenizer"),
            "--record", str(RECORDS / "train.json")]
        run = run_bounded(command, timeout=remaining, log=LOGS / "train.log", monitor_every=1800)
        RUNS["train"] = run.record()
        print(run.summary())
        ranks = [json.loads(p.read_text()) for p in sorted(RECORDS.glob("train.rank*.json"))]
        saved = sorted(RUN_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        TRAIN_FACTS = {"outcome": [r["outcome"] for r in ranks], "global_step": [r["global_step"] for r in ranks],
                       "checkpoints": [p.name for p in saved], "exports": sorted(p.name for p in RUN_DIR.glob("export-*")),
                       "mean_step_seconds": [r["mean_step_seconds_after_two"] for r in ranks],
                       "peak_reserved_gib": [r["peak_reserved_gib"] for r in ranks]}
        GATES.record("train", run.ok and bool(saved), json.dumps(TRAIN_FACTS))


In [ ]:
# The latest full checkpoint (for the next session) and every float16 export
# (for choosing on the development set) are kept; older checkpoints were
# rotated away by the trainer.
KEEP = Path("/kaggle/working/run")
KEEP.mkdir(exist_ok=True)
saved = sorted(Path("/tmp/run").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1])) if Path("/tmp/run").is_dir() else []
if saved:
    shutil.copytree(saved[-1], KEEP / saved[-1].name, dirs_exist_ok=True)
for export in sorted(Path("/tmp/run").glob("export-*")) if Path("/tmp/run").is_dir() else []:
    shutil.copytree(export, KEEP / export.name, dirs_exist_ok=True)
for name in ("train.log", "initial_config.json"):
    if (Path("/tmp/run") / name).is_file():
        shutil.copy2(Path("/tmp/run") / name, KEEP / name)
if (Path("/tmp/run") / "tensorboard").is_dir():
    shutil.copytree(Path("/tmp/run") / "tensorboard", KEEP / "tensorboard", dirs_exist_ok=True)
destination = Path("/kaggle/working/session")
destination.mkdir(exist_ok=True)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
for path in RECORDS.glob("*.json"):
    shutil.copy2(path, destination / path.name)
total = sum(p.stat().st_size for p in Path("/kaggle/working").rglob("*") if p.is_file())
print("kept:", [p.name for p in KEEP.iterdir()], f"- {total / 2**30:.1f} GiB in /kaggle/working")
print("gates:", json.dumps(GATES.summary(), indent=1))
